# DreamSnap — Free Generation Worker

Keeps a GPU notebook running as a **worker**: it polls your backend for pending
image / pack jobs, renders them with Flux + the model's LoRA, uploads the results,
and marks them `COMPLETED`. Users keep using the app normally — clicking *Generate*
queues a job that this worker picks up.

## Before you start
1. Backend must run in self-hosted mode: set `TRAINING_MODE=manual` and a
   `WORKER_SECRET` env var on Render.
2. Runtime → **Change runtime type → GPU**.
3. Hugging Face token with access to `black-forest-labs/FLUX.1-dev`.

> Keep this tab open while you want generation to work. Free Colab sessions last
> ~12 h (Kaggle ~9 h). When it stops, jobs just stay `PENDING` until you run it again.
>
> Flux on a free T4 is slow — expect **1–3 min per image** with CPU offload.

## 1. Configure

In [ ]:
BACKEND_URL   = "https://dreamsnap.onrender.com"   # your deployed backend
WORKER_SECRET = "PASTE_WORKER_SECRET"              # must match WORKER_SECRET on the backend
HF_TOKEN      = "hf_xxxxxxxxxxxxxxxx"               # token with FLUX.1-dev access

NUM_STEPS     = 25      # 20-30; lower = faster
GUIDANCE      = 3.5
WIDTH, HEIGHT = 768, 1024
POLL_SECONDS  = 8       # how often to check for new jobs

## 2. Install

In [ ]:
!nvidia-smi -L
!pip install -q -U diffusers transformers accelerate peft sentencepiece protobuf requests
!pip install -q optimum-quanto

## 3. Load Flux (quantized + CPU offload so it fits a T4)

In [ ]:
import torch, requests, io, time
from huggingface_hub import login
from diffusers import FluxPipeline

login(token=HF_TOKEN)

pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    torch_dtype=torch.bfloat16,
)

# 8-bit quantize the big transformer + T5 so it fits in ~16 GB, then offload to CPU.
try:
    from optimum.quanto import freeze, quantize, qfloat8
    quantize(pipe.transformer, weights=qfloat8); freeze(pipe.transformer)
    quantize(pipe.text_encoder_2, weights=qfloat8); freeze(pipe.text_encoder_2)
    print("quantized to 8-bit")
except Exception as e:
    print("quantization skipped:", e)

pipe.enable_model_cpu_offload()
print("pipeline ready")

## 4. Helpers (load LoRA, upload image)

In [ ]:
HEADERS = {"x-worker-secret": WORKER_SECRET}
_loaded_lora = {"url": None}

def ensure_lora(url):
    """Download + load the LoRA weights for a job, caching the current one."""
    if _loaded_lora["url"] == url:
        return
    r = requests.get(url, timeout=300); r.raise_for_status()
    open("/content/lora.safetensors", "wb").write(r.content)
    try:
        pipe.unload_lora_weights()
    except Exception:
        pass
    pipe.load_lora_weights("/content/lora.safetensors")
    _loaded_lora["url"] = url
    print("loaded LoRA:", url.split("/")[-1])

def upload_image(img):
    """Upload a PIL image to S3 via the backend presigned URL, return public URL."""
    buf = io.BytesIO(); img.save(buf, format="PNG"); buf.seek(0)
    g = requests.post(f"{BACKEND_URL}/api/get-upload-url",
                      json={"fileName": "gen.png", "fileType": "image/png"}, timeout=60)
    g.raise_for_status(); up = g.json()
    requests.put(up["uploadURL"], data=buf.getvalue(),
                 headers={"Content-Type": "image/png"}, timeout=600).raise_for_status()
    return up["publicURL"]

def complete(job, image_urls=None, failed=False):
    ep = "image" if job["type"] == "image" else "packimage"
    body = {"failed": True} if failed else {"imageUrls": image_urls}
    requests.post(f"{BACKEND_URL}/worker/jobs/{ep}/{job['id']}/complete",
                  json=body, headers=HEADERS, timeout=60).raise_for_status()

## 5. Run the worker loop (keep this running)

In [ ]:
seen = set()  # in-memory de-dupe (single worker)
print("worker started; polling", BACKEND_URL)

while True:
    try:
        jr = requests.get(f"{BACKEND_URL}/worker/jobs", headers=HEADERS, timeout=60)
        jr.raise_for_status()
        jobs = [j for j in jr.json().get("jobs", []) if j["id"] not in seen]
        if not jobs:
            time.sleep(POLL_SECONDS); continue

        for job in jobs:
            seen.add(job["id"])
            try:
                print(f"-> {job['type']} {job['id']}: {job['prompt'][:60]}")
                ensure_lora(job["loraUrl"])
                image = pipe(
                    job["prompt"],
                    num_inference_steps=NUM_STEPS,
                    guidance_scale=GUIDANCE,
                    width=WIDTH, height=HEIGHT,
                ).images[0]
                url = upload_image(image)
                complete(job, image_urls=[url])
                print("   done:", url)
            except Exception as e:
                print("   job failed:", e)
                try: complete(job, failed=True)
                except Exception as e2: print("   (could not mark failed:", e2, ")")
    except KeyboardInterrupt:
        print("stopped"); break
    except Exception as e:
        print("poll error:", e); time.sleep(POLL_SECONDS * 2)